# ArSL Word Training v2 — Improved Architecture (Standalone / Independent)

**Improvements over v1:**

| # | Change | Reason |
|---|--------|--------|
| 1 | `TimeDistributed` Dense spatial encoder per frame | CNN+BiLSTM hybrid achieves 99% on KArSL vs 94% raw landmarks |
| 2 | `MultiHeadAttention` (4 heads) replacing simple attention | Attends to multiple motion patterns simultaneously |
| 3 | Sequence length 30 → **48 frames** | Captures full sign arc; evenly-sampled frames outperform dense consecutive |
| 4 | `GRU(64)` layer added after BiLSTM stack | BiLSTM+GRU combo shown superior to pure BiLSTM |
| 5 | Face features dropped — **258 features** (Pose+Hands only) | Face 204 features add noise; hands+pose carry all discriminative signal |
| 6 | Cosine Annealing LR with warm restarts | Finds better minima vs plateau-reactive ReduceLROnPlateau |
| 7 | Horizontal flip augmentation (swap LH↔RH) | Doubles effective data; forces hand-shape-agnostic learning |

### Feature Vector (258 features/frame):

| Stream     | Landmarks            | Features |
|------------|----------------------|----------|
| Pose       | 33 × (x,y,z,vis)    | 132      |
| Left Hand  | 21 × (x,y,z)        | 63       |
| Right Hand | 21 × (x,y,z)        | 63       |
| **Total**  |                      | **258**  |

### Pipeline:
1. GPU Detection & Configuration
2. Config & Paths
3. Load Shared Vocabulary
4. Helper Functions (extract / pad / augment)
5. Build Dataset (or load `.npz` cache)
6. Data Exploration
7. Preprocessing & Splits
8. Build & Train improved model
9. Evaluation Dashboard

### How to Run:
1. Set `PROJECT_ROOT` in Cell 3 if your drive differs
2. Set `USE_PREEXTRACTED_KEYPOINTS = False` for raw `.mp4` videos
3. Run all cells top-to-bottom
4. After first run the `.npz` cache is saved — subsequent runs are instant

For a **custom word subset** (demo / basic vocabulary), use `ArSL_Word_Training_CustomWords.ipynb` after the full NPZ is built.


In [1]:
# ============================================================
# Cell 1: Imports
# ============================================================
import os
import time
import warnings
from pathlib import Path

# Must be set BEFORE TensorFlow loads cuBLAS, otherwise the MX150 (and any
# small-VRAM GPU) will fail the very first MatMul with a cublas error.
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'
os.environ['TF_CPP_MIN_LOG_LEVEL']      = '2'   # hide INFO + WARNING spam
os.environ['TF_GPU_ALLOCATOR']          = 'cuda_malloc_async'

import cv2
import mediapipe as mp_lib
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, classification_report
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, LSTM, GRU, Bidirectional, Dense, Dropout,
    BatchNormalization, TimeDistributed, MultiHeadAttention,
    GlobalAveragePooling1D, Add, LayerNormalization
)
from tensorflow.keras.callbacks import (
    ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
)
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import mixed_precision
from tqdm import tqdm

warnings.filterwarnings('ignore', category=UserWarning)

print('=' * 60)
print('✅ All libraries imported successfully!')
print(f'📦 TensorFlow : {tf.__version__}')
print(f'📦 NumPy      : {np.__version__}')
print(f'📦 Pandas     : {pd.__version__}')
print(f'📦 OpenCV     : {cv2.__version__}')
print('=' * 60)


✅ All libraries imported successfully!
📦 TensorFlow : 2.10.0
📦 NumPy      : 1.23.5
📦 Pandas     : 2.0.3
📦 OpenCV     : 4.11.0


In [2]:
# ============================================================
# Cell 2: GPU Detection & Configuration  (cuBLAS-safe)
# ============================================================
from tensorflow.python.platform import build_info as _tf_build_info

print('=' * 60)
print('🔍 GPU DETECTION & CONFIGURATION')
print('=' * 60)
print(f'   TF version    : {tf.__version__}')
print(f'   TF CUDA build : {_tf_build_info.build_info.get("cuda_version",  "n/a")}')
print(f'   TF cuDNN build: {_tf_build_info.build_info.get("cudnn_version", "n/a")}')

gpus = tf.config.list_physical_devices('GPU')
USE_GPU = False
DEVICE  = '/CPU:0'

if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        tf.config.set_visible_devices(gpus[0], 'GPU')
        USE_GPU = True
        DEVICE  = '/GPU:0'
        print(f'✅ GPU visible : {gpus[0].name}')
        try:
            details = tf.config.experimental.get_device_details(gpus[0])
            print(f'   Device     : {details.get("device_name", "unknown")}')
            print(f'   CUDA CC    : {details.get("compute_capability", "unknown")}')
        except Exception:
            pass
    except RuntimeError as e:
        print(f'⚠️  GPU config error: {e}')
else:
    print('⚠️  No GPU detected — training on CPU (slower)')

mixed_precision.set_global_policy('float32')
print(f'📐 Precision   : float32 (stable for LSTM/GRU)')

# ── cuBLAS smoke test  (graceful fallback on MX150 / mixed CUDA installs) ──
if USE_GPU:
    try:
        with tf.device('/GPU:0'):
            _a = tf.constant([[1.0, 2.0], [3.0, 4.0]])
            _b = tf.constant([[5.0, 6.0], [7.0, 8.0]])
            _c = tf.matmul(_a, _b)
            _ = _c.numpy()  # force kernel launch + sync
        print(f'✅ GPU compute test passed: {_c.device}')
    except Exception as e:
        USE_GPU = False
        DEVICE  = '/CPU:0'
        tf.config.set_visible_devices([], 'GPU')   # hide the GPU from Keras
        print('⚠️  GPU compute test FAILED — falling back to CPU (training will continue).')
        print(f'   Reason : {type(e).__name__}: {str(e)[:200]}')
        print('   Likely causes (one of):')
        print('   1. cuBLAS / cuDNN version mismatch with TF 2.10 (needs CUDA 11.2 + cuDNN 8.1).')
        print('   2. Another process is holding the GPU (Chrome HW-accel, another notebook).')
        print('   3. MX150 2 GB VRAM exhausted by a previous run — restart the kernel.')
        print('   Quick fixes:')
        print('   - Restart kernel and re-run from Cell 1 (env vars at top of Cell 1 apply on next launch).')
        print('   - Close all browsers and other GPU-using apps, then retry.')
        print('   - On Kaggle: switch the notebook to a GPU runtime (no driver work needed).')

print(f'\n✅ Using device: {DEVICE}')
print('=' * 60)


🔍 GPU DETECTION & CONFIGURATION
   TF version    : 2.10.0
   TF CUDA build : 64_112
   TF cuDNN build: 64_8
✅ GPU visible : /physical_device:GPU:0
   Device     : NVIDIA GeForce MX150
   CUDA CC    : (6, 1)
📐 Precision   : float32 (stable for LSTM/GRU)
✅ GPU compute test passed: /job:localhost/replica:0/task:0/device:GPU:0

✅ Using device: /GPU:0


In [3]:
# ============================================================
# Cell 3: Configuration & Paths  ← only change PROJECT_ROOT
# ============================================================
PROJECT_ROOT = Path(r'M:/Term 10/Grad')

SLR_MAIN    = PROJECT_ROOT / 'SLR Main'
WORDS_ROOT  = SLR_MAIN / 'Words'
KARSL_ROOT  = Path(r'E:\Downloads\Arabic Words Dataset')   # ← dataset location
OUTPUT_DIR  = WORDS_ROOT / 'ArSL Word (Arabic)'
LABELS_FILE = OUTPUT_DIR / 'KARSL-502_Labels.txt'          # class ID → Arabic/English names
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── v2 Feature Layout: Pose + Hands ONLY (no face) ─────────
#   Pose      : 33 × 4 (x,y,z,vis) = 132
#   Left hand : 21 × 3             =  63
#   Right hand: 21 × 3             =  63
#   TOTAL                          = 258
POSE_FEATURES = 33 * 4   # 132
HAND_FEATURES = 21 * 3   #  63 per hand
NUM_FEATURES  = POSE_FEATURES + HAND_FEATURES * 2  # 258

# ── Hyperparameters ─────────────────────────────────────────
SEQUENCE_LENGTH = 48        # ↑ from 30 — captures full sign arc
BATCH_SIZE      = 32        # reduced — 502 classes + MX150 2GB VRAM
EPOCHS          = 200
LEARNING_RATE   = 5e-4
LSTM_UNITS_1    = 128       # reduced — only 8 samples/class, smaller model generalizes better
LSTM_UNITS_2    = 96
GRU_UNITS       = 64
SPATIAL_ENC_1   = 192       # TimeDistributed encoder dim 1
SPATIAL_ENC_2   = 128       # TimeDistributed encoder dim 2
DENSE_UNITS     = 384
DROPOUT_RATE    = 0.4       # higher dropout — very few samples per class, needs strong regularization
LABEL_SMOOTH    = 0.1
GRAD_CLIP_NORM  = 1.0
TEST_SIZE       = 0.4
MHA_HEADS       = 4         # Multi-Head Attention heads
MHA_KEY_DIM     = 32        # reduced key dim to match smaller model

# Set True to load pre-extracted .npy/.csv keypoints (fast)
# Set False to extract from raw .mp4 using MediaPipe (first run)
USE_PREEXTRACTED_KEYPOINTS = False
print('=' * 60)
print('⚙️  ArSL WORD TRAINING v2 — STANDALONE CONFIGURATION')
print('=' * 60)
for name, path in [('KArSL root', KARSL_ROOT), ('Labels file', LABELS_FILE)]:
    status = '✅' if path.exists() else '❌ NOT FOUND'
    print(f'{status} {name}: {path}')
print(f'\n📁 Output dir      : {OUTPUT_DIR}')
print(f'📐 Features/frame  : {NUM_FEATURES}  (Pose {POSE_FEATURES} + L.Hand {HAND_FEATURES} + R.Hand {HAND_FEATURES})')
print(f'⚙️  Sequence length  : {SEQUENCE_LENGTH} frames')
print(f'⚙️  Batch size       : {BATCH_SIZE}')
print(f'⚙️  Max epochs       : {EPOCHS}')
print(f'⚙️  Learning rate    : {LEARNING_RATE}')
print(f'⚙️  Pre-extracted    : {USE_PREEXTRACTED_KEYPOINTS}')
print(f'🆕 MHA heads        : {MHA_HEADS} × key_dim {MHA_KEY_DIM}')
print(f'🆕 Spatial encoder  : {SPATIAL_ENC_1} → {SPATIAL_ENC_2}')
print(f'🆕 GRU units        : {GRU_UNITS}')


⚙️  ArSL WORD TRAINING v2 — STANDALONE CONFIGURATION
✅ KArSL root: E:\Downloads\Arabic Words Dataset
✅ Labels file: M:\Term 10\Grad\SLR Main\Words\ArSL Word (Arabic)\KARSL-502_Labels.txt

📁 Output dir      : M:\Term 10\Grad\SLR Main\Words\ArSL Word (Arabic)
📐 Features/frame  : 258  (Pose 132 + L.Hand 63 + R.Hand 63)
⚙️  Sequence length  : 48 frames
⚙️  Batch size       : 32
⚙️  Max epochs       : 200
⚙️  Learning rate    : 0.0005
⚙️  Pre-extracted    : False
🆕 MHA heads        : 4 × key_dim 32
🆕 Spatial encoder  : 192 → 128
🆕 GRU units        : 64


In [4]:
# ============================================================
# Cell 4: Load KArSL Labels (Standalone — no shared vocab)
# ============================================================
print('=' * 60)
print('📚 LOADING KARSL LABELS')
print('=' * 60)

id_to_english = {}
id_to_arabic  = {}

if LABELS_FILE.exists():
    with open(str(LABELS_FILE), 'r', encoding='utf-8', errors='replace') as fh:
        for line in fh:
            line = line.strip()
            if not line or line.lower().startswith('signid'):
                continue
            parts = line.split('\t')
            if len(parts) >= 3:
                try:
                    sid = int(parts[0])
                    ar  = parts[1].strip()
                    en  = parts[2].strip()
                    mapped_id = sid + 1
                    id_to_english[mapped_id] = en if en and en not in ('?', '??', '') else str(mapped_id)
                    id_to_arabic[mapped_id]  = ar if ar and ar not in ('?', '??', '') else en
                except Exception:
                    continue
    print(f'✅ Labels loaded: {len(id_to_english)} entries')
    print(f'   Sample: {list(id_to_english.items())[:5]}')
else:
    print('⚠️  Labels file not found — numeric class IDs will be used as names')
    print(f'   Expected at: {LABELS_FILE}')

# target_karsl_classes is built in Cell 6 by scanning the actual dataset folders
print('\n📌 Class list will be built by scanning KArSL dataset folders in Cell 6')


📚 LOADING KARSL LABELS
✅ Labels loaded: 502 entries
   Sample: [(2, '0'), (3, '1'), (4, '2'), (5, '3'), (6, '4')]

📌 Class list will be built by scanning KArSL dataset folders in Cell 6


In [5]:
# ============================================================
# Cell 5: Helper Functions
# ============================================================

def build_feature_vector(results):
    """
    Pack MediaPipe Holistic results into a flat (258,) vector.
    v2: Pose(132) + LeftHand(63) + RightHand(63) — face removed.
    """
    # Pose: 33 × (x, y, z, visibility) = 132
    if results.pose_landmarks:
        pose = np.array(
            [[lm.x, lm.y, lm.z, lm.visibility]
             for lm in results.pose_landmarks.landmark],
            dtype=np.float32
        ).flatten()
    else:
        pose = np.zeros(POSE_FEATURES, dtype=np.float32)

    # Left hand: 21 × (x, y, z) = 63
    if results.left_hand_landmarks:
        lh = np.array(
            [[lm.x, lm.y, lm.z] for lm in results.left_hand_landmarks.landmark],
            dtype=np.float32
        ).flatten()
    else:
        lh = np.zeros(HAND_FEATURES, dtype=np.float32)

    # Right hand: 21 × (x, y, z) = 63
    if results.right_hand_landmarks:
        rh = np.array(
            [[lm.x, lm.y, lm.z] for lm in results.right_hand_landmarks.landmark],
            dtype=np.float32
        ).flatten()
    else:
        rh = np.zeros(HAND_FEATURES, dtype=np.float32)

    return np.concatenate([pose, lh, rh])  # (258,)


def pad_or_sample(sequence, target_len=SEQUENCE_LENGTH, target_feat=NUM_FEATURES):
    """Pad (short) or uniformly sample (long) a sequence to (target_len, target_feat)."""
    arr = np.array(sequence, dtype=np.float32)
    if arr.ndim != 2:
        return None

    # Fix feature dim
    if arr.shape[1] > target_feat:
        arr = arr[:, :target_feat]
    elif arr.shape[1] < target_feat:
        arr = np.concatenate(
            [arr, np.zeros((arr.shape[0], target_feat - arr.shape[1]), dtype=np.float32)],
            axis=1
        )

    # Fix time dim — evenly spaced sampling (better than consecutive truncation)
    if arr.shape[0] >= target_len:
        idx = np.linspace(0, arr.shape[0] - 1, target_len, dtype=int)
        arr = arr[idx]
    else:
        pad = np.zeros((target_len - arr.shape[0], target_feat), dtype=np.float32)
        arr = np.concatenate([arr, pad], axis=0)

    return arr  # (SEQUENCE_LENGTH, NUM_FEATURES)


def extract_from_video(video_path, holistic=None):
    """Extract MediaPipe Holistic landmarks from a raw .mp4 file."""
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return None

    frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = holistic.process(rgb)
        frames.append(build_feature_vector(results))

    cap.release()

    if not frames:
        return None
    return pad_or_sample(np.array(frames, dtype=np.float32))


def create_holistic():
    """Single reusable MediaPipe Holistic instance (model_complexity=0 saves RAM)."""
    return mp_lib.solutions.holistic.Holistic(
        static_image_mode=False,
        model_complexity=0,
        enable_segmentation=False,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5,
    )


print('✅ Helper functions ready')
print(f'   Feature vector: Pose({POSE_FEATURES}) + L.Hand({HAND_FEATURES}) + R.Hand({HAND_FEATURES}) = {NUM_FEATURES}')


✅ Helper functions ready
   Feature vector: Pose(132) + L.Hand(63) + R.Hand(63) = 258


In [6]:
# ============================================================
# Cell 6: Build Dataset (or Load .npz Cache)
# Standalone — discovers all classes by scanning KArSL_ROOT
# ============================================================
print('=' * 60)
print('📦 BUILDING ARABIC WORD DATASET (STANDALONE)')
print('=' * 60)

NPZ_PATH = OUTPUT_DIR / 'arsl_word_sequences_v2_full.npz'   # always the full cache
LEGACY_NPZ = OUTPUT_DIR / 'arsl_word_sequences_v2.npz'      # old partial cache
ARCHIVE_DIR = OUTPUT_DIR / '_archive' / 'stale_outputs'
ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)
MIN_SAMPLES_CACHE = 15000
X = y = None

def archive_bad_cache(path, reason):
    """Move unusable cache aside instead of loading it into RAM."""
    dest = ARCHIVE_DIR / path.name
    if dest.exists():
        dest = ARCHIVE_DIR / f'{path.stem}_{int(time.time())}{path.suffix}'
    try:
        path.rename(dest)
        print(f'   📁 Moved bad cache → {dest}')
        print(f'   Reason: {reason}')
        return True
    except OSError:
        print(f'   ⚠️  Locked cache (ignored): {path.name} — {reason}')
        return False

def try_load_cache(path):
    """Return (X, y) if path is a valid full cache, else None."""
    if not path.exists():
        return None
    print(f'\n💾 Checking cache: {path.name}')
    try:
        with np.load(str(path), mmap_mode='r') as _d:
            if 'X' not in _d.files or 'y' not in _d.files:
                archive_bad_cache(path, 'missing X or y arrays')
                return None
            x_shape = _d['X'].shape
            if len(x_shape) != 3 or x_shape[1] != SEQUENCE_LENGTH or x_shape[2] != NUM_FEATURES:
                archive_bad_cache(path, f'bad shape {x_shape}')
                return None
            if x_shape[0] < MIN_SAMPLES_CACHE:
                archive_bad_cache(
                    path,
                    f'only {x_shape[0]} samples (<{MIN_SAMPLES_CACHE} — test split only)',
                )
                return None
            print(f'   Valid cache shape (mmap): {x_shape}')
        _d = np.load(str(path))
        return _d['X'], _d['y']
    except MemoryError as e:
        archive_bad_cache(path, f'MemoryError: {e}')
        return None
    except (OSError, ValueError, KeyError) as e:
        archive_bad_cache(path, f'{type(e).__name__}: {e}')
        return None

for _cache in (NPZ_PATH, LEGACY_NPZ):
    loaded = try_load_cache(_cache)
    if loaded is not None:
        X, y = loaded
        print(f'   X shape : {X.shape}')
        print(f'   y shape : {y.shape}')
        print(f'   Classes : {len(np.unique(y))}')
        target_karsl_classes = sorted(np.unique(y).tolist())
        for cid in target_karsl_classes:
            id_to_english.setdefault(cid, str(cid))
            id_to_arabic.setdefault(cid, str(cid))
        print('   ✅ Loaded from cache — skipping extraction')
        break

if X is None:
    if not KARSL_ROOT.exists():
        raise FileNotFoundError(
            f'KArSL dataset not found at: {KARSL_ROOT}\n'
            f'Download from: https://www.kaggle.com/datasets/yousefelkilany/karsl-502'
        )

    # ── Print actual folder structure for diagnostics ────────
    print(f'\n🔍 Scanning dataset: {KARSL_ROOT}')
    print('\n📂 Top-level contents:')
    top_entries = sorted(KARSL_ROOT.iterdir(), key=lambda p: p.name)
    for e in top_entries[:20]:
        marker = '📁' if e.is_dir() else '📄'
        print(f'   {marker} {e.name}')
    if len(top_entries) > 20:
        print(f'   ... and {len(top_entries) - 20} more')

    # ── Discover numeric class folders at any depth (up to 3 levels) ──
    all_class_dirs = {}  # class_id (int) → Path

    def try_add(path):
        """Try to register a folder as a class directory if name is numeric."""
        raw = path.name.lstrip('0') or '0'
        try:
            cid = int(raw)
            all_class_dirs.setdefault(cid, []).append(path)
        except ValueError:
            pass

    # Level 1: direct children
    for entry in top_entries:
        if entry.is_dir():
            try_add(entry)

    # Level 2: grandchildren (e.g. dataset/split/class_id/)
    if not all_class_dirs:
        for entry in top_entries:
            if entry.is_dir():
                for sub in sorted(entry.iterdir()):
                    if sub.is_dir():
                        try_add(sub)

    # Level 3: great-grandchildren (e.g. dataset/signer/split/class_id/)
    if not all_class_dirs:
        for entry in top_entries:
            if entry.is_dir():
                for sub in entry.iterdir():
                    if sub.is_dir():
                        for subsub in sub.iterdir():
                            if subsub.is_dir():
                                try_add(subsub)

    if not all_class_dirs:
        print('\n❌ Could not find numeric class folders.')
        print('   Folder structure found:')
        for e in top_entries[:5]:
            if e.is_dir():
                children = list(e.iterdir())[:5]
                for c in children:
                    print(f'      {e.name}/{c.name}')
        raise FileNotFoundError(
            f'No numeric class folders found in {KARSL_ROOT}\n'
            f'Please check the printed structure above and update KARSL_ROOT to point\n'
            f'directly to the folder that contains numbered subfolders (e.g. 001, 002...)'
        )

    target_karsl_classes = sorted(all_class_dirs.keys())
    for cid in target_karsl_classes:
        id_to_english.setdefault(cid, str(cid))
        id_to_arabic.setdefault(cid,  str(cid))

    print(f'   Found {len(target_karsl_classes)} class folders')
    print(f'   Class ID range: {target_karsl_classes[0]} → {target_karsl_classes[-1]}')
    named = sum(1 for c in target_karsl_classes if id_to_english[c] != str(c))
    print(f'   Named classes  : {named} / {len(target_karsl_classes)}')

    # ── Extract sequences (CPU-only — release TF GPU first) ───
    import gc
    print(f'\n⏳ Extracting from: {KARSL_ROOT}')
    print(f'   Mode : {"Pre-extracted .npy/.csv" if USE_PREEXTRACTED_KEYPOINTS else "Raw .mp4 via MediaPipe"}')
    print('   Releasing TensorFlow GPU memory (MediaPipe runs on CPU)...')
    tf.keras.backend.clear_session()
    try:
        tf.config.set_visible_devices([], 'GPU')
    except Exception:
        pass
    gc.collect()

    start_time = time.time()
    X_list, y_list = [], []
    empty_classes = 0
    CHECKPOINT_EVERY = 50
    CHECKPOINT_PATH = OUTPUT_DIR / 'arsl_word_sequences_v2_partial.npz'

    holistic = None if USE_PREEXTRACTED_KEYPOINTS else create_holistic()
    try:
        for ci, class_id in enumerate(tqdm(target_karsl_classes, desc='Loading classes')):
            class_dirs = all_class_dirs[class_id]
            SKIP_EXTS = {'.zip', '.rar', '.7z', '.tar', '.gz'}

            if USE_PREEXTRACTED_KEYPOINTS:
                files = [f for cd in class_dirs
                         for f in list(cd.rglob('*.npy')) + list(cd.rglob('*.csv'))
                         if f.suffix.lower() not in SKIP_EXTS]
            else:
                files = [f for cd in class_dirs
                         for f in cd.rglob('*.mp4')
                         if f.suffix.lower() not in SKIP_EXTS]
            if not files:
                files = [f for cd in class_dirs
                         for f in (list(cd.rglob('*.npy')) +
                                   list(cd.rglob('*.csv')) +
                                   list(cd.rglob('*.mp4')))
                         if f.suffix.lower() not in SKIP_EXTS]
            if not files:
                empty_classes += 1
                continue

            for fp in files:
                seq = None
                try:
                    if fp.suffix.lower() == '.npy':
                        seq = pad_or_sample(np.load(fp))
                    elif fp.suffix.lower() == '.csv':
                        seq = pad_or_sample(pd.read_csv(fp).values)
                    elif fp.suffix.lower() == '.mp4':
                        seq = extract_from_video(fp, holistic)
                except Exception:
                    continue
                if seq is None:
                    continue
                if np.sum(np.all(seq == 0, axis=1)) / len(seq) > 0.8:
                    continue
                X_list.append(seq)
                y_list.append(class_id)

            if holistic and (ci + 1) % CHECKPOINT_EVERY == 0 and X_list:
                np.savez_compressed(str(CHECKPOINT_PATH),
                                    X=np.array(X_list, dtype=np.float32),
                                    y=np.array(y_list, dtype=np.int32))
                gc.collect()
                print(f'\n   💾 Checkpoint: {len(X_list):,} samples saved')
    finally:
        if holistic is not None:
            holistic.close()
        gc.collect()

    elapsed = time.time() - start_time
    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list,  dtype=np.int32)

    print(f'\n✅ Done in {elapsed:.1f}s ({elapsed/60:.1f} min)')
    print(f'   X shape        : {X.shape}')
    print(f'   Classes loaded : {len(target_karsl_classes) - empty_classes} / {len(target_karsl_classes)}')
    print(f'   Empty classes  : {empty_classes}')

    np.savez_compressed(NPZ_PATH, X=X, y=y)
    print(f'\n💾 Saved cache: {NPZ_PATH}')
    if CHECKPOINT_PATH.exists():
        CHECKPOINT_PATH.unlink()

    # Re-enable GPU for training cells (re-run Cell 2 if training fails)
    if globals().get('USE_GPU'):
        try:
            gpus = tf.config.list_physical_devices('GPU')
            if gpus:
                tf.config.set_visible_devices(gpus[0], 'GPU')
                print('GPU re-enabled for training')
        except Exception:
            pass

assert X is not None and y is not None, 'Cell 6 failed — X/y not built. Re-run this cell and wait for extraction to finish.'
assert X.shape[1] == SEQUENCE_LENGTH and X.shape[2] == NUM_FEATURES
unique_ids, counts = np.unique(y, return_counts=True)
word_names = [id_to_english.get(int(uid), str(uid)) for uid in unique_ids]
print(f'\nDataset ready: {X.shape[0]:,} samples, {len(unique_ids)} classes')
print(f'  Min/class: {counts.min()}  Max/class: {counts.max()}  Mean: {counts.mean():.1f}')


📦 BUILDING ARABIC WORD DATASET (STANDALONE)

🔍 Scanning dataset: E:\Downloads\Arabic Words Dataset

📂 Top-level contents:
   📄 class_distribution_check.png
   📁 test
   📁 train


   Found 502 class folders
   Class ID range: 1 → 502
   Named classes  : 501 / 502

⏳ Extracting from: E:\Downloads\Arabic Words Dataset
   Mode : Raw .mp4 via MediaPipe
   Releasing TensorFlow GPU memory (MediaPipe runs on CPU)...


Loading classes:   2%|▏         | 12/502 [20:40<14:04:15, 103.38s/it]


KeyboardInterrupt: 

In [ ]:
# ============================================================
# Cell 7: Data Exploration
# ============================================================
if 'y' not in globals() or y is None:
    raise RuntimeError('Run Cell 6 first — dataset (X, y) is not loaded yet.')

print('=' * 60)
print('📊 DATA EXPLORATION')
print('=' * 60)

unique_ids, counts = np.unique(y, return_counts=True)
word_names_en = [id_to_english.get(int(uid), str(uid)) for uid in unique_ids]

sort_idx      = np.argsort(counts)[::-1]
sorted_names  = [word_names_en[i] for i in sort_idx]
sorted_counts = counts[sort_idx]

fig, axes = plt.subplots(1, 2, figsize=(22, 5))

axes[0].bar(range(len(sorted_names)), sorted_counts, color='darkgreen', edgecolor='black', linewidth=0.3)
axes[0].set_xticks(range(len(sorted_names)))
axes[0].set_xticklabels(sorted_names, rotation=90, fontsize=5)
axes[0].set_title(f'Class Distribution — {len(unique_ids)} classes, {len(y)} samples', fontsize=13)
axes[0].axhline(np.mean(sorted_counts), color='red',    linestyle='--', alpha=0.7, label=f'Mean {np.mean(sorted_counts):.0f}')
axes[0].axhline(np.median(sorted_counts), color='orange', linestyle=':',  alpha=0.7, label=f'Median {np.median(sorted_counts):.0f}')
axes[0].legend(fontsize=9)

axes[1].hist(sorted_counts, bins=25, color='darkgreen', edgecolor='black', alpha=0.85)
axes[1].set_xlabel('Samples per Class', fontsize=11)
axes[1].set_ylabel('Number of Classes', fontsize=11)
axes[1].set_title('How Many Classes Have N Samples?', fontsize=13)
axes[1].axvline(np.mean(sorted_counts), color='red', linestyle='--', label=f'Mean {np.mean(sorted_counts):.0f}')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

print(f'\n📊 Summary:')
print(f'   Total samples    : {len(y)}')
print(f'   Total classes    : {len(unique_ids)}')
print(f'   Min samples/class: {counts.min()} ({word_names_en[counts.argmin()]})')
print(f'   Max samples/class: {counts.max()} ({word_names_en[counts.argmax()]})')
print(f'   Mean / Median    : {counts.mean():.1f} / {np.median(counts):.1f}')

low = [(word_names_en[i], counts[i]) for i in range(len(counts)) if counts[i] < 5]
if low:
    print(f'\n⚠️  Classes with <5 samples ({len(low)}): {", ".join(f"{n}({c})" for n,c in low)}')


In [ ]:
# ============================================================
# Cell 7: Preprocessing & Splits
# ============================================================
print('=' * 60)
print('🔧 PREPROCESSING & TRAIN/VAL/TEST SPLIT')
print('=' * 60)

_d  = np.load(str(NPZ_PATH))
X, y = _d['X'], _d['y']

# StandardScaler — fit on flattened, reshape back
orig_shape = X.shape
X_flat     = X.reshape(-1, NUM_FEATURES)
scaler     = StandardScaler()
X_flat     = scaler.fit_transform(X_flat)
X          = X_flat.reshape(orig_shape).astype(np.float32)

# Save scaler for inference
np.savez_compressed(
    str(OUTPUT_DIR / 'arsl_v2_scaler.npz'),
    mean=scaler.mean_.astype(np.float32),
    scale=scaler.scale_.astype(np.float32)
)
print('✅ StandardScaler applied and saved')

# Encode labels
encoder     = LabelEncoder()
y_encoded   = encoder.fit_transform(y)
num_classes = len(encoder.classes_)
y_onehot    = to_categorical(y_encoded, num_classes=num_classes)

# Save class map
class_df = pd.DataFrame({
    'model_class_index': range(num_classes),
    'karsl_class_id'   : encoder.classes_.tolist(),
    'english'          : [id_to_english.get(int(c), str(c)) for c in encoder.classes_],
    'arabic'           : [id_to_arabic.get(int(c),  str(c)) for c in encoder.classes_],
})
class_df.to_csv(OUTPUT_DIR / 'arsl_v2_classes.csv', index=False)
print(f'✅ Class map saved ({num_classes} classes)')
print(class_df.head(10).to_string())

# Stratified 60/20/20 split
try:
    X_train, X_tmp, y_train, y_tmp = train_test_split(
        X, y_onehot, test_size=TEST_SIZE, random_state=42, stratify=y_encoded
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_tmp, y_tmp, test_size=0.5, random_state=42, stratify=np.argmax(y_tmp, 1)
    )
except ValueError:
    print('⚠️  Stratified split failed — using random split')
    X_train, X_tmp, y_train, y_tmp = train_test_split(X, y_onehot, test_size=TEST_SIZE, random_state=42)
    X_val,  X_test, y_val,  y_test = train_test_split(X_tmp, y_tmp, test_size=0.5, random_state=42)

# Balanced class weights
train_ints = np.argmax(y_train, axis=1)
cw_array   = compute_class_weight('balanced', classes=np.arange(num_classes), y=train_ints)
cw_array   = np.clip(cw_array, 0.5, 10.0)
class_weights = dict(enumerate(cw_array))

print(f'\n📊 Split Summary:')
print(f'   Train       : {X_train.shape[0]}  ({X_train.shape[0]/len(X)*100:.0f}%)')
print(f'   Validation  : {X_val.shape[0]}  ({X_val.shape[0]/len(X)*100:.0f}%)')
print(f'   Test        : {X_test.shape[0]}  ({X_test.shape[0]/len(X)*100:.0f}%)')
print(f'   Classes     : {num_classes}')
print(f'   Input shape : {X_train.shape[1:]}')


In [ ]:
# ============================================================
# Cell 8: Build & Train — Improved Architecture
#
# Architecture:
#   Input (48, 258)
#   → TimeDistributed Dense 256 + BN     [spatial encoder]
#   → TimeDistributed Dense 128 + BN
#   → BiLSTM(192) + BN + SpatialDropout
#   → BiLSTM(128) + BN + SpatialDropout
#   → GRU(64)     + BN                   [final temporal encoder]
#   → MultiHeadAttention(4 heads, key=64) [attend to key frames]
#   → GlobalAveragePooling1D
#   → Dense(256) + BN + Dropout
#   → Dense(128) + Dropout
#   → Softmax(num_classes)
# ============================================================
print('=' * 60)
print('🚀 BUILDING & TRAINING IMPROVED MODEL')
print('=' * 60)

tf.keras.backend.clear_session()
BATCH_SIZE_TRAIN = BATCH_SIZE  # always respect Cell 3

# ── Augmentation function ────────────────────────────────────
POSE_F = tf.constant(POSE_FEATURES, dtype=tf.int32)
HAND_F = tf.constant(HAND_FEATURES, dtype=tf.int32)

def augment_sequence(x, y_label):
    # 1) Gaussian noise
    x = x + tf.random.normal(tf.shape(x), mean=0.0, stddev=0.005)
    # 2) Temporal shift ±3 frames
    x = tf.roll(x, shift=tf.random.uniform([], -3, 4, dtype=tf.int32), axis=0)
    # 3) Frame dropout ~10%
    mask = tf.cast(tf.random.uniform([SEQUENCE_LENGTH, 1]) > 0.1, tf.float32)
    x = x * mask
    # 4) Random scale (simulate signer distance variation)
    x = x * tf.random.uniform([], 0.9, 1.1)
    # 5) Horizontal flip — swap left/right hand blocks
    #    layout: pose[0:132] | lh[132:195] | rh[195:258]
    if tf.random.uniform([]) > 0.5:
        pose = x[:, :POSE_F]
        lh   = x[:, POSE_F : POSE_F + HAND_F]
        rh   = x[:, POSE_F + HAND_F :]
        x    = tf.concat([pose, rh, lh], axis=-1)  # swap lh↔rh
    return x, y_label

# ── tf.data pipelines ───────────────────────────────────────
AUTOTUNE = tf.data.AUTOTUNE

train_ds = (
    tf.data.Dataset.from_tensor_slices((X_train, y_train))
    .shuffle(min(len(X_train), 10000), seed=42, reshuffle_each_iteration=True)
    .map(augment_sequence, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE_TRAIN)
    .prefetch(AUTOTUNE)
)
val_ds = (
    tf.data.Dataset.from_tensor_slices((X_val, y_val))
    .batch(BATCH_SIZE_TRAIN)
    .prefetch(AUTOTUNE)
)
test_ds = (
    tf.data.Dataset.from_tensor_slices((X_test, y_test))
    .batch(BATCH_SIZE_TRAIN)
    .prefetch(AUTOTUNE)
)
print(f'✅ tf.data pipelines ready (augmentation: noise + shift + frame-drop + scale + LR-flip)')

# ── Build Model (4GB VRAM — full v2 architecture) ──────────────
# Architecture:
#   Input (48, 258)
#   TimeDistributed Dense 192->128  [per-frame spatial encoder]
#   BiLSTM(128)  + BN + SpatialDropout
#   BiLSTM(96)   + BN + SpatialDropout
#   LSTM(64)     + BN   return_seq=False   [compact final encoding]
#   Dense(384)   + BN + Dropout
#   Dense(192)   + Dropout
#   Softmax(num_classes)
# recurrent_dropout=0 everywhere -> cuDNN-accelerated LSTM

inputs = Input(shape=(SEQUENCE_LENGTH, NUM_FEATURES), name='landmark_input')

# Per-frame spatial encoder
x = TimeDistributed(Dense(SPATIAL_ENC_1, activation='relu',
                           kernel_initializer='he_normal',
                           kernel_regularizer=tf.keras.regularizers.l2(1e-4)),
                    name='td_enc_1')(inputs)
x = TimeDistributed(BatchNormalization(), name='td_bn_1')(x)
x = TimeDistributed(Dropout(0.2), name='td_drop_1')(x)
x = TimeDistributed(Dense(SPATIAL_ENC_2, activation='relu',
                           kernel_initializer='he_normal',
                           kernel_regularizer=tf.keras.regularizers.l2(1e-4)),
                    name='td_enc_2')(x)
x = TimeDistributed(BatchNormalization(), name='td_bn_2')(x)

# Bidirectional temporal layers
x = Bidirectional(LSTM(LSTM_UNITS_1, return_sequences=True,
                        kernel_regularizer=tf.keras.regularizers.l2(1e-4)),
                  name='bilstm_1')(x)
x = BatchNormalization(name='bn_1')(x)
x = tf.keras.layers.SpatialDropout1D(DROPOUT_RATE, name='sdrop_1')(x)

x = Bidirectional(LSTM(LSTM_UNITS_2, return_sequences=True,
                        kernel_regularizer=tf.keras.regularizers.l2(1e-4)),
                  name='bilstm_2')(x)
x = BatchNormalization(name='bn_2')(x)
x = tf.keras.layers.SpatialDropout1D(DROPOUT_RATE, name='sdrop_2')(x)

# Final LSTM - return last state only (memory efficient vs GAP)
x = LSTM(GRU_UNITS, return_sequences=False,
          kernel_regularizer=tf.keras.regularizers.l2(1e-4),
          name='lstm_final')(x)
x = BatchNormalization(name='bn_lstm_final')(x)
x = Dropout(DROPOUT_RATE, name='drop_lstm')(x)

# Classifier head
x = Dense(DENSE_UNITS, activation='relu',
           kernel_initializer='he_normal',
           kernel_regularizer=tf.keras.regularizers.l2(1e-4),
           name='dense_1')(x)
x = BatchNormalization(name='bn_dense_1')(x)
x = Dropout(DROPOUT_RATE, name='drop_1')(x)

x = Dense(DENSE_UNITS // 2, activation='relu',
           kernel_initializer='he_normal',
           kernel_regularizer=tf.keras.regularizers.l2(1e-4),
           name='dense_2')(x)
x = Dropout(DROPOUT_RATE * 0.5, name='drop_2')(x)

outputs = Dense(num_classes, activation='softmax', dtype='float32', name='output')(x)

model = Model(inputs, outputs, name='ArSL_Word_v2_4GB')

# ── Cosine Annealing LR ──────────────────────────────────────
total_steps      = (len(X_train) // BATCH_SIZE_TRAIN) * EPOCHS
first_decay_steps = total_steps // 5
lr_schedule = tf.keras.optimizers.schedules.CosineDecayRestarts(
    initial_learning_rate=LEARNING_RATE,
    first_decay_steps=max(first_decay_steps, 1),
    t_mul=2.0,
    m_mul=0.9,
    alpha=1e-7
)

optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule, clipnorm=GRAD_CLIP_NORM)
loss_fn   = tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTH)

model.compile(
    optimizer=optimizer,
    loss=loss_fn,
    metrics=['accuracy', tf.keras.metrics.TopKCategoricalAccuracy(k=5, name='top5_acc')]
)

print('\n📐 Model Architecture:')
model.summary()
print(f'\n🖥️  Training on: {DEVICE}')

# ── Callbacks ────────────────────────────────────────────────
MODEL_BEST  = str(OUTPUT_DIR / 'arsl_v2_best.h5')
MODEL_FINAL = str(OUTPUT_DIR / 'arsl_v2_final.h5')

callbacks = [
    ModelCheckpoint(MODEL_BEST, monitor='val_accuracy', save_best_only=True, mode='max', verbose=1),
    EarlyStopping(monitor='val_loss', patience=25, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.TerminateOnNaN(),
]

# ── Train ─────────────────────────────────────────────────────
print(f'\n🚀 Starting training...')
print(f'   Device        : {DEVICE}')
print(f'   Batch size    : {BATCH_SIZE_TRAIN}')
print(f'   Max epochs    : {EPOCHS}')
print(f'   LR schedule   : Cosine Annealing with warm restarts')
print(f'   Label smooth  : {LABEL_SMOOTH}')
print(f'   Grad clip     : {GRAD_CLIP_NORM}')
print(f'   Class weights : balanced, clipped [0.5, 10]')
start_time = time.time()

with tf.device(DEVICE):
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=callbacks,
        class_weight=class_weights,
        verbose=1
    )

elapsed = time.time() - start_time
print(f'\n✅ Training complete in {elapsed:.1f}s ({elapsed/60:.1f} min)')
print(f'   Best val_accuracy : {max(history.history["val_accuracy"]):.4f}')
print(f'   Best val_top5_acc : {max(history.history["val_top5_acc"]):.4f}')

model.save(MODEL_FINAL)
print(f'\n💾 Best model  : {MODEL_BEST}')
print(f'💾 Final model : {MODEL_FINAL}')
print(f'💾 Class map   : {OUTPUT_DIR / "arsl_v2_classes.csv"}')
print(f'💾 Scaler      : {OUTPUT_DIR / "arsl_v2_scaler.npz"}')


In [ ]:
# ============================================================
# Cell 9: Evaluation Dashboard
# ============================================================
print('=' * 60)
print('📈 EVALUATION DASHBOARD')
print('=' * 60)

best_model = tf.keras.models.load_model(MODEL_BEST)

with tf.device(DEVICE):
    proba = best_model.predict(test_ds, verbose=0)

y_pred = np.argmax(proba, axis=1)
y_true = np.argmax(y_test, axis=1)

top1_acc = (y_pred == y_true).mean()
top5_acc = sum(
    1 for i in range(len(y_true)) if y_true[i] in np.argsort(proba[i])[-5:]
) / len(y_true)

print(f'\n🎯 Test Results:')
print(f'   Top-1 Accuracy : {top1_acc:.4f} ({top1_acc*100:.2f}%)')
print(f'   Top-5 Accuracy : {top5_acc:.4f} ({top5_acc*100:.2f}%)')
print(f'   Test samples   : {len(y_true)}')
print(f'   Classes        : {num_classes}')

word_labels = [
    id_to_english.get(int(encoder.classes_[i]), str(encoder.classes_[i]))
    for i in range(num_classes)
]

# ── Plot 1: Training curves ──────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(22, 5))

axes[0].plot(history.history['accuracy'],     label='Train', linewidth=2, color='#2E7D32')
axes[0].plot(history.history['val_accuracy'], label='Val',   linewidth=2, color='#FF9800')
axes[0].set_title('Accuracy', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(True, alpha=0.3); axes[0].set_ylim([0, 1.05])
best_ep = np.argmax(history.history['val_accuracy'])
axes[0].axvline(best_ep, color='blue', linestyle=':', alpha=0.5, label=f'Best epoch {best_ep}')

axes[1].plot(history.history['loss'],     label='Train', linewidth=2, color='#2E7D32')
axes[1].plot(history.history['val_loss'], label='Val',   linewidth=2, color='#FF9800')
axes[1].set_title('Loss', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].plot(history.history['top5_acc'],     label='Train Top-5', linewidth=2, color='#2E7D32')
axes[2].plot(history.history['val_top5_acc'], label='Val Top-5',   linewidth=2, color='#FF9800')
axes[2].set_title('Top-5 Accuracy', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Top-5 Accuracy')
axes[2].legend(); axes[2].grid(True, alpha=0.3); axes[2].set_ylim([0, 1.05])

plt.suptitle(f'ArSL v2 Training — Top-1: {top1_acc*100:.1f}%  Top-5: {top5_acc*100:.1f}%',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'arsl_v2_training_curves.png'), dpi=150)
plt.show()

# ── Plot 2: Overfitting monitor ──────────────────────────────
gap = np.array(history.history['accuracy']) - np.array(history.history['val_accuracy'])
fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(range(len(gap)),
       gap,
       color=['green' if g < 0.05 else 'orange' if g < 0.15 else 'red' for g in gap],
       edgecolor='black', linewidth=0.3, alpha=0.8)
ax.axhline(0.05, color='green', linestyle='--', alpha=0.5, label='Healthy gap (5%)')
ax.axhline(0.15, color='red',   linestyle='--', alpha=0.5, label='Overfitting (15%)')
ax.set_title('Overfitting Monitor (Train − Val Accuracy)', fontsize=13, fontweight='bold')
ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy Gap')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ── Plot 3: Per-class F1 bar chart ───────────────────────────
report = classification_report(
    y_true, y_pred, target_names=word_labels, zero_division=0, output_dict=True
)
print('\n📋 Classification Report:')
print(classification_report(y_true, y_pred, target_names=word_labels, zero_division=0))

class_f1 = {k: v['f1-score'] for k, v in report.items() if k in word_labels}
sorted_f1 = sorted(class_f1.items(), key=lambda x: x[1], reverse=True)
f1_names, f1_vals = zip(*sorted_f1) if sorted_f1 else ([], [])

fig, ax = plt.subplots(figsize=(24, 6))
colors_f1 = ['#4CAF50' if v >= 0.7 else '#FF9800' if v >= 0.4 else '#F44336' for v in f1_vals]
ax.bar(range(len(f1_names)), f1_vals, color=colors_f1, edgecolor='black', linewidth=0.3)
ax.set_xticks(range(len(f1_names)))
ax.set_xticklabels(f1_names, rotation=90, fontsize=5)
ax.axhline(np.mean(f1_vals), color='blue', linestyle='--', alpha=0.5, label=f'Mean F1: {np.mean(f1_vals):.3f}')
ax.set_title(f'Per-Class F1 (green≥0.7, orange≥0.4, red<0.4) — Mean: {np.mean(f1_vals):.3f}', fontsize=13)
ax.set_ylim([0, 1.05]); ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'arsl_v2_f1_scores.png'), dpi=150)
plt.show()

# ── Plot 4: Confusion matrix ─────────────────────────────────
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(20, 18))
sns.heatmap(
    cm,
    annot=(num_classes <= 50),
    fmt='d' if num_classes <= 50 else '',
    cmap='Greens',
    xticklabels=word_labels,
    yticklabels=word_labels,
    ax=ax
)
ax.set_title(f'Confusion Matrix — {num_classes} classes  Top-1: {top1_acc*100:.1f}%', fontsize=14)
ax.set_xlabel('Predicted', fontsize=12); ax.set_ylabel('True', fontsize=12)
plt.xticks(rotation=90, fontsize=5); plt.yticks(fontsize=5)
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'arsl_v2_confusion_matrix.png'), dpi=150)
plt.show()

# ── Plot 5: Top-10 confused pairs ───────────────────────────
cm_nodiag = cm.copy(); np.fill_diagonal(cm_nodiag, 0)
confused = sorted(
    [(word_labels[i], word_labels[j], cm_nodiag[i, j])
     for i in range(num_classes) for j in range(num_classes) if cm_nodiag[i, j] > 0],
    key=lambda x: x[2], reverse=True
)[:10]
if confused:
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.barh([f'{p[0]} → {p[1]}' for p in confused],
            [p[2] for p in confused],
            color='#E91E63', edgecolor='darkred', alpha=0.85)
    ax.set_xlabel('Misclassification Count', fontsize=12)
    ax.set_title('Top-10 Most Confused Pairs (True → Predicted)', fontsize=13, fontweight='bold')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

# ── Plot 6: Best & Worst performing classes ──────────────────
per_class_acc = {}
for i in range(num_classes):
    mask = y_true == i
    if mask.sum() > 0:
        per_class_acc[word_labels[i]] = (y_pred[mask] == i).mean()

sorted_acc = sorted(per_class_acc.items(), key=lambda x: x[1])
n_show = min(10, len(sorted_acc))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))

worst = sorted_acc[:n_show]
ax1.barh(range(len(worst)), [w[1]*100 for w in worst], color='#F44336', edgecolor='darkred', alpha=0.85)
ax1.set_yticks(range(len(worst)))
ax1.set_yticklabels([w[0] for w in worst], fontsize=10)
ax1.set_xlabel('Accuracy (%)', fontsize=12)
ax1.set_title(f'Bottom {n_show} Classes', fontsize=13, fontweight='bold', color='#F44336')
ax1.set_xlim([0, 105])
for i, w in enumerate(worst):
    ax1.text(w[1]*100 + 1, i, f'{w[1]*100:.1f}%', va='center', fontsize=10)

best = sorted_acc[-n_show:][::-1]
ax2.barh(range(len(best)), [b[1]*100 for b in best], color='#4CAF50', edgecolor='darkgreen', alpha=0.85)
ax2.set_yticks(range(len(best)))
ax2.set_yticklabels([b[0] for b in best], fontsize=10)
ax2.set_xlabel('Accuracy (%)', fontsize=12)
ax2.set_title(f'Top {n_show} Classes', fontsize=13, fontweight='bold', color='#4CAF50')
ax2.set_xlim([0, 105])
for i, b in enumerate(best):
    ax2.text(b[1]*100 + 1, i, f'{b[1]*100:.1f}%', va='center', fontsize=10)

plt.suptitle('Best vs Worst Performing Classes', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'arsl_v2_best_worst_classes.png'), dpi=150)
plt.show()

print('\n' + '=' * 60)
print('✅ Evaluation complete!')
print(f'   Top-1: {top1_acc*100:.2f}%  |  Top-5: {top5_acc*100:.2f}%')
print('=' * 60)


## Tips & Troubleshooting

| Issue | Solution |
|-------|----------|
| **OOM (Out of Memory)** | Reduce `BATCH_SIZE` to 32 or 16 |
| **No GPU detected** | Check CUDA/cuDNN installation |
| **Low accuracy (<70%)** | Check dataset path and that videos loaded correctly in Cell 6 |
| **Labels show as numbers** | Check `KARSL-502_Labels.txt` exists in the ArSL Word folder |
| **No class folders found** | Ensure KArSL_502 subfolders are named numerically (e.g. `0001`) |
| **NaN loss** | Lower `LEARNING_RATE` to `1e-4`, set `LABEL_SMOOTH=0` |
| **Slow extraction** | Only happens once — cache `.npz` is saved automatically |
| **Want to re-extract** | Delete `arsl_word_sequences_v2.npz` and re-run Cell 6 |

### Monitor GPU during training:
```powershell
nvidia-smi -l 1
```

### v2 vs v1 Architecture Differences:

| Component | v1 | v2 |
|-----------|----|-----------|
| Features | 462 (Pose+Face+Hands) | **258 (Pose+Hands only)** |
| Sequence length | 30 | **48** |
| Spatial encoding | None (raw landmarks → LSTM) | **TimeDistributed Dense 256→128** |
| Attention | Custom single-head | **MultiHeadAttention (4 heads)** |
| Temporal layers | BiLSTM → BiLSTM → LSTM | **BiLSTM → BiLSTM → GRU** |
| Attention pooling | Custom reduce_sum | **Residual + LayerNorm + GAP** |
| LR schedule | ReduceLROnPlateau | **Cosine Annealing with warm restarts** |
| Augmentation | 4 types | **5 types (+ horizontal flip)** |
